# BEIP — EDA 01: Election Results Analysis

**Objective:** Explore the core election results from `silver.election_results` to understand:
- Historical candidate volume across election years
- Party dynamics (candidacy numbers, total wins, and party win rates)
- Vote share distributions for winners vs. non-winners
- Voter turnout patterns across Indian states
- Target variable distribution (`won`) and class imbalance

## 1. Setup and Data Loading

In [4]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))
from src.config import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM silver.election_results", engine)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")

OperationalError: (psycopg2.OperationalError) connection to server at "localhost" (::1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?

(Background on this error at: https://sqlalche.me/e/20/e3q8)

## 2. Basic Dataset Overview
Check data types, column null counts, and summary statistics.

In [2]:
df.head()

NameError: name 'df' is not defined

In [ ]:
print("=== Column Data Types ===")
print(df.dtypes)

print("\n=== Missing Values Count ===")
print(df.isnull().sum())

In [ ]:
df.describe()

## 3. Election Year Distribution
How many candidates ran across different election years?

In [ ]:
year_counts = df.groupby("year").size().reset_index(name="candidate_count")
display(year_counts)

plt.figure(figsize=(10, 5))
sns.barplot(data=year_counts, x="year", y="candidate_count", color="steelblue")
plt.title("Total Candidates per Election Year", fontsize=14, fontweight="bold")
plt.xlabel("Election Year")
plt.ylabel("Number of Candidates")
plt.show()

## 4. Party Performance and Win Rates
Evaluate party representation and calculate conversion rates (wins / contested seats).

In [ ]:
# Top 20 parties by candidates fielded
top_parties = df["party"].value_counts().head(20)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_parties.values, y=top_parties.index, color="teal")
plt.title("Top 20 Parties by Number of Candidates Fielded", fontsize=14, fontweight="bold")
plt.xlabel("Total Candidates")
plt.ylabel("Party")
plt.show()

In [ ]:
# Calculate Win Rates for top parties with at least 50 candidates
party_stats = df.groupby("party").agg(
    total_candidates=("candidate_name", "count"),
    total_wins=("position", lambda p: (p == 1).sum()),
    avg_vote_share=("vote_share", "mean")
).reset_index()

party_stats["win_rate_pct"] = (party_stats["total_wins"] / party_stats["total_candidates"]) * 100

# Filter to parties with substantial candidacy to avoid outlier 100% win-rates (e.g. min 50 candidates)
major_party_stats = party_stats[party_stats["total_candidates"] >= 50].sort_values("win_rate_pct", ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(data=major_party_stats, x="win_rate_pct", y="party", palette="viridis")
plt.title("Win Rate (%) for Major Parties (Min 50 Candidates)", fontsize=14, fontweight="bold")
plt.xlabel("Win Rate (%)")
plt.ylabel("Party")
plt.show()

display(major_party_stats)

## 5. Vote Share Distribution
Compare vote share distributions for winners (position = 1) versus non-winners.

In [ ]:
df["is_winner"] = df["position"].apply(lambda p: "Winner (Pos 1)" if p == 1 else "Non-Winner")

plt.figure(figsize=(12, 6))
sns.histplot(data=df, x="vote_share", hue="is_winner", bins=40, kde=True, stat="density", common_norm=False)
plt.title("Vote Share Distribution: Winners vs. Non-Winners", fontsize=14, fontweight="bold")
plt.xlabel("Vote Share (%)")
plt.ylabel("Density")
plt.show()

# Summary statistics for winners
winner_summary = df[df["position"] == 1]["vote_share"].describe()
print("=== Vote Share Summary for Winners ===")
print(winner_summary)

## 6. Voter Turnout by State
Analyze which states consistently record high vs. low voter turnout.

In [ ]:
state_turnout = df.groupby("state_name")["turnout_percentage"].mean().reset_index()
state_turnout = state_turnout.dropna().sort_values("turnout_percentage", ascending=False)

plt.figure(figsize=(12, 10))
sns.barplot(data=state_turnout, x="turnout_percentage", y="state_name", color="cornflowerblue")
plt.title("Average Voter Turnout (%) by State", fontsize=14, fontweight="bold")
plt.xlabel("Average Turnout (%)")
plt.ylabel("State")
plt.show()

## 7. Target Variable & Class Imbalance
Define the machine learning target `won` and examine the class balance ratio.

In [ ]:
df["won"] = (df["position"] == 1).astype(int)
won_counts = df["won"].value_counts()

print("=== Target Variable Counts (won) ===")
print(f"Lost (0): {won_counts.get(0, 0):,} ({won_counts.get(0, 0)/len(df)*100:.2f}%)")
print(f"Won  (1): {won_counts.get(1, 0):,} ({won_counts.get(1, 0)/len(df)*100:.2f}%)")
print(f"Imbalance Ratio: ~{won_counts.get(0, 0) // max(1, won_counts.get(1, 0))}:1")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="won", palette=["#e74c3c", "#2ecc71"])
plt.title("Target Class Distribution (0 = Lost, 1 = Won)", fontsize=12, fontweight="bold")
plt.xticks([0, 1], ["Lost (0)", "Won (1)"])
plt.ylabel("Count")
plt.show()